In [34]:
# 필요한 라이브러리 
import pandas as pd
import numpy as np
from datetime import timedelta
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# 예제 트랜젝션 데이터 생성
np.random.seed(42)

customers = ['C' + str(i) for i in range(1, 11)]
data = []

for cust in customers:
    purchase_dates = pd.to_datetime('2023-01-01') + pd.to_timedelta(
        np.random.randint(0, 180, size = np.random.randint(2,5)),
        unit='D'
    )
    purchase_dates = sorted(purchase_dates)
    for d in purchase_dates:
        amount = np.random.randint(10000, 50000) # 1~5만원 랜덤 구매 금액
        data.append([cust, d, amount])

df = pd.DataFrame(data, columns = ['customer_id', 'purchase_date', 'amount'])

In [35]:
## 피처 정리가 되어야 한다.
## BG/NBD 모델 (고객생존 + 제구매)
## Gamma-Gamma 모델 학습 (고객평균 구매금액 추정)
## 향후 기간 6개원(180일) 기대 구매 횟수 예측
## 평균 구매 예측
## LTV = 기대 구매 횟수 X 평균 구매 금액
## 라벨링된 결과가 나온다

In [36]:
df

,customer_id,purchase_date,amount
0,C1,2023-01-15,21284
1,C1,2023-04-03,16265
2,C1,2023-04-17,26850
3,C1,2023-06-29,47194
4,C2,2023-03-29,26023
5,C2,2023-04-10,11685
6,C2,2023-04-14,10769
7,C2,2023-04-27,12433
8,C3,2023-02-27,38693
9,C3,2023-06-10,16396


In [40]:
## 관측치 날짜 종료일 선택
observation_date = pd.to_datetime('2023-09-01')

summary = summary_data_from_transaction_data(
    transactions = df,
    customer_id_col = 'customer_id',
    datetime_col = 'purchase_date',
    monetary_value_col = 'amount',
    observation_period_end = observation_date,
    freq = 'D'
)

#### frequency - 첫 구매 이후 재구매 횟수 (최소 1회이상)
#### recency - 첫 구매 이후 마지막 구매일까지 경과일
#### T - 첫 구매 이후 관측 종료일까지의 기간
#### monetary_value - 고객당 평균 거래 금액 (평균 단가)

In [41]:
summary

,frequency,recency,T,monetary_value
customer_id,,,,
C1,3.0,165.0,229.0,30103.000000
C10,2.0,81.0,230.0,37489.000000
C2,3.0,29.0,156.0,11629.000000
C3,1.0,103.0,186.0,16396.000000
C4,1.0,10.0,195.0,28431.000000
C5,3.0,124.0,193.0,25500.333333
C6,3.0,146.0,223.0,19284.000000
C7,2.0,81.0,235.0,25850.000000
C8,1.0,114.0,194.0,33939.000000


In [45]:
## BG/NBD 모델(고객 생존 + 재구매) 학습
## 고객의 생존 여부와 재구매 횟수를 모델링하는 패키지
## 과적합 막기 위한 정규화 항
## frequency, recency, T
bgf = BetaGeoFitter(penalizer_coef = 0.01)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

<lifetimes.BetaGeoFitter: fitted with 10 subjects, a: 1.16, alpha: 117.21, b: 1.83, r: 2.43>

In [48]:
## Gamma-Gamma 모델 학습(고객 평균 구매금액 추정)
## frequncy, monetory_value
ggf = GammaGammaFitter(penalizer_coef = 0.01)
ggf.fit(summary['frequency'], summary['monetary_value'])

<lifetimes.GammaGammaFitter: fitted with 10 subjects, p: 2.65, q: 0.13, v: 2.52>

In [50]:
## 향후 기간 6개월(180일) 기대 구매 횟수 예측
summary['expected_purchases_6m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    180,
    summary['frequency'],
    summary['recency'],
    summary['T']
)

In [52]:
## 평균 구매 금액
summary['expected_avg_value'] = ggf.conditional_expected_average_profit(summary['frequency'], summary['monetary_value'])

In [53]:
## LTV 계산
summary['expected_LTV_6m'] = summary['expected_purchases_6m'] * summary['expected_avg_value']

In [55]:
summary

,frequency,recency,T,monetary_value,expected_purchases_6m,expected_avg_value,expected_LTV_6m
customer_id,,,,,,,
C1,3.0,165.0,229.0,30103.000000,1.128197,33821.048726,38156.813097
C10,2.0,81.0,230.0,37489.000000,0.297155,44891.107408,13339.614216
C2,3.0,29.0,156.0,11629.000000,0.258444,13065.887595,3376.804050
C3,1.0,103.0,186.0,16396.000000,0.517752,24465.340860,12666.970764
C4,1.0,10.0,195.0,28431.000000,0.099284,42420.642076,4211.686971
C5,3.0,124.0,193.0,25500.333333,1.077117,28650.046869,30859.458189
C6,3.0,146.0,223.0,19284.000000,0.988479,21666.123518,21416.515390
C7,2.0,81.0,235.0,25850.000000,0.278726,30954.485075,8627.833269
C8,1.0,114.0,194.0,33939.000000,0.533889,50638.157581,27035.174505
